In [ ]:
%load_ext autoreload
%autoreload 2

# Import

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Union
from glob import glob

import sys
PHASE1_SRC = Path("../Phase1/src/")
if str(PHASE1_SRC) not in sys.path:
    sys.path.insert(0, str(PHASE1_SRC))

from space import Space, GFLOWNET_ENV
from reward import reward_peak, reward_latent
from machinelearning import train_single_model, predict_with_model, preprocess, build_models
from VEM import oracle_fn, TARGET_NAMES
from plotting import *

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from umap import UMAP
from sklearn.decomposition import PCA


# Loading

In [ ]:
run = "grid_results_2"

In [ ]:
grid = pd.read_csv(f"{run}/grid_summary.csv")#.sort_values("run")

In [ ]:
grid.head()

In [ ]:
from glob import glob
paths = sorted(glob(f"{run}/*/al_metrics.csv"))
paths = [p for p in paths if Path(p).parent.name in grid["run"].values] # filters per reward as wreitten above
if not paths:
    raise ValueError("No files found at al_results/*/al_metrics.csv")
df = pd.concat([load(p).assign(run=Path(p).parent.name) for p in paths], ignore_index=True)

In [ ]:
df

In [ ]:
df[df["sampling_strategy"]=="gflownet"].groupby("run").count().count().iloc[0]

In [ ]:
best_runs(df)

## Visualization

In [ ]:
metrics = attach_grid_metadata(df, grid)

In [ ]:
metrics.head()

In [ ]:
select_best_runs(metrics, metric="best_reward")

In [ ]:
plot_reward_curves(select_best_runs(metrics, metric="mean_reward"), metric="mean_reward_this_iter", title="Mean reward per iteration", ceiling=True)

In [ ]:
aggregate_seeds(metrics, "mean_reward_this_iter", groupby=("sampling_strategy",)).groupby(["sampling_strategy", "iteration"]).agg(
    mean=("mean", "mean"),
    std=("std", "mean"),
    sem=("sem", "mean"),
    ci95=("ci95", "mean"),
)

In [ ]:
plot_seed_variance(metrics, metric="mean_reward_this_iter")

In [ ]:
plot_oracle_efficiency(metrics, metric="mean_reward_this_iter")

In [ ]:
plot_proxy_curves(metrics)

In [ ]:
plot_final_boxplot(metrics, metric="best_reward")

In [ ]:
plot_reward_heatmap(metrics, metric="best_reward")

In [ ]:
plot_budget_tradeoff(metrics, metric="mean_reward")

In [ ]:
embedding, preprocessor, reducer = compute_global_embedding(
    method="umap"
)

In [ ]:
sampled_dict_best = {}
for samp in df["sampling_strategy"].unique():
    run_id = best_runs(df).loc[best_runs(df)["sampling_strategy"] == samp, "run"].values[0]
    run_df = pd.read_csv(f"{run}/{run_id}/dataset.csv")
    sampled_dict_best[samp] = prepare_run_embedding(run_df, preprocessor, reducer)

In [ ]:
plot_sampling_grid(sampled_dict_best, cols=2, colorscale="Plasma")

In [ ]:
plot_sampling_grid(sampled_dict_best, cols=2, colorscale="Plasma")

In [ ]:
plot_exploration_dashboard(sampled_dict_best, preprocessor)

In [ ]:
plot_peak_coverage(sampled_dict_best, "inverse")

In [ ]:
plot_peak_coverage_heatmap(sampled_dict_best)

In [ ]:
sampled_dict_all = {}
for samp in df["sampling_strategy"].unique():
    run_ids = df.loc[df["sampling_strategy"] == samp, "run"].unique()
    pooled = pd.concat(
        [pd.read_csv(f"{run}/{r}/dataset.csv") for r in run_ids],
        ignore_index=True,
    )
    sampled_dict_all[samp] = prepare_run_embedding(pooled, preprocessor, reducer)
    

In [ ]:
plot_exploration_dashboard(sampled_dict_all, preprocessor)

In [ ]:
plot_peak_coverage(sampled_dict_all, 'inverse')

In [ ]:
plot_peak_coverage_heatmap(sampled_dict_all)

In [ ]:
df = all_metrics[all_metrics["iteration"] > 0]


# plot
fig = go.Figure()
for i, val in enumerate(sorted(df["gfn_loss"].dropna().unique())):
    agg = df[df["gfn_loss"] == val].groupby("iteration")["mean_reward_this_iter"].agg(["mean","std"]).reset_index()
    c = f"hsl({i * 60}, 70%, 50%)"
    fig.add_trace(go.Scatter(x=agg["iteration"], y=agg["mean"], name=str(val),
                                line=dict(color=c, width=2), mode="lines+markers"))
    fig.add_trace(go.Scatter(
        x=pd.concat([agg["iteration"], agg["iteration"][::-1]]),
        y=pd.concat([agg["mean"]+agg["std"], (agg["mean"]-agg["std"])[::-1]]),
        fill="toself", fillcolor=c, opacity=0.15, line=dict(width=0),
        showlegend=False, hoverinfo="skip"))
fig.update_layout(title=f"{'mean_reward_this_iter'} by {'gfn_loss'}", xaxis_title="iteration",
                    plot_bgcolor="white", hovermode="x unified")